# Лабораторная работа №1. Синтез управляющего конечного автомата

Егорова Варвара Александровна, P3323, 408575

Требуется спроектировать управляющий конечный автомат:
- определить возможные состояния (не менее 5)  и переходы между ними,
- определить входные события, инициирующие переходы, и выходные сигналы.

Построить диаграмму переходов автомата.
Реализовать работу конечного автомата на языке Python (с использованием библиотеки transitions или без нее).

****
**Диаграмма состояний**

Карта не вставлена - карта вставлена - ввод ПИН - выбор операции - выполнение операции (снять/пополнить) - прием/выдача денег - отдать карту - завершение

In [ ]:
%pip install transitions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 4.4 MB/s eta 0:00:00


In [ ]:
import time
from transitions import Machine

class ATM:
    def __init__(self):
        self.states = [
            'initial',
            'card_inserted',
            'pin_waiting',
            'operation_waiting',
            'executing',
            'money_receiving',
            'money_issuing',
            'card_returning',
            'completed',
            'defect'
        ]

        self.transitions = [
            {'trigger': 'insert_card_trigger', 'source': 'initial', 'dest': 'card_inserted'},
            {'trigger': 'card_read_success', 'source': 'card_inserted', 'dest': 'pin_waiting'},
            {'trigger': 'card_read_failure', 'source': 'card_inserted', 'dest': 'defect'},
            {'trigger': 'pin_entered', 'source': 'pin_waiting', 'dest': 'operation_waiting'},
            {'trigger': 'pin_error', 'source': 'pin_waiting', 'dest': 'card_returning'},
            {'trigger': 'operation_selected', 'source': 'operation_waiting', 'dest': 'executing'},
            {'trigger': 'cancel', 'source': 'operation_waiting', 'dest': 'card_returning'},
            {'trigger': 'withdrawal_selected', 'source': 'executing', 'dest': 'money_issuing'},
            {'trigger': 'deposit_selected', 'source': 'executing', 'dest': 'money_receiving'},
            {'trigger': 'operation_failure', 'source': 'executing', 'dest': 'defect'},
            {'trigger': 'money_dispensed', 'source': 'money_issuing', 'dest': 'card_returning'},
            {'trigger': 'dispense_failure', 'source': 'money_issuing', 'dest': 'defect'},
            {'trigger': 'money_received', 'source': 'money_receiving', 'dest': 'card_returning'},
            {'trigger': 'receive_failure', 'source': 'money_receiving', 'dest': 'defect'},
            {'trigger': 'card_returned', 'source': 'card_returning', 'dest': 'completed'},
            {'trigger': 'return_failure', 'source': 'card_returning', 'dest': 'defect'},
            {'trigger': 'repair', 'source': 'defect', 'dest': 'initial'},
            {'trigger': 'reset', 'source': 'completed', 'dest': 'initial'}
        ]

        self.machine = Machine(model=self, states=self.states,
                              transitions=self.transitions, initial='initial')

        self.card_inserted = False
        self.pin_attempts = 0
        self.max_pin_attempts = 3
        self.card_data = None
        self.operation_type = None
        self.amount = 0
        self.balance = 10000
        self.atm_cash = 50000

    def insert_card(self, card_data):
        if self.state != 'initial':
            print("Ошибка: Банкомат не готов к приему карты!")
            return False

        print("Карта вставлена. Чтение данных...")
        self.card_inserted = True
        self.card_data = card_data
        self.pin_attempts = 0

        self.insert_card_trigger()

        time.sleep(1)
        if self.simulate_card_reading():
            print("Карта успешно считана.")
            self.card_read_success()
            return True
        else:
            print("Ошибка чтения карты!")
            self.card_read_failure()
            return False

    def simulate_card_reading(self):
        return True

    def enter_pin(self, pin):
        if self.state != 'pin_waiting':
            print(f"Ошибка: Банкомат не ожидает ввод ПИН-кода! Текущее состояние: {self.state}")
            return False

        print(f"Проверка ПИН-кода...")
        time.sleep(1)

        if self.verify_pin(pin):
            print("ПИН-код принят.")
            self.pin_entered()
            return True
        else:
            self.pin_attempts += 1
            remaining = self.max_pin_attempts - self.pin_attempts
            if remaining > 0:
                print(f"Неверный ПИН-код. Осталось попыток: {remaining}")
                return False
            else:
                print("Превышено количество попыток. Карта будет возвращена.")
                self.pin_error()
                return False

    def verify_pin(self, pin):
        return True

    def select_operation(self, operation):
        if self.state != 'operation_waiting':
            print(f"Ошибка: Банкомат не ожидает выбора операции! Текущее состояние: {self.state}")
            return False

        self.operation_type = operation
        print(f"Выбрана операция: {operation}")
        self.operation_selected()
        return True

    def cancel_operation(self):
        if self.state == 'operation_waiting':
            print("Операция отменена. Карта возвращается...")
            self.cancel()
            return True
        return False

    def execute_withdrawal(self, amount):
        if self.state != 'executing':
            print(f"Ошибка: Банкомат не готов к выполнению операции! Текущее состояние: {self.state}")
            return False

        self.amount = amount
        print(f"Запрошено снятие: {amount} руб.")
        time.sleep(1)

        if amount <= self.balance and amount <= self.atm_cash:
            self.balance -= amount
            self.atm_cash -= amount
            print(f"Операция одобрена. Новый баланс: {self.balance} руб.")
            self.withdrawal_selected()
            self.dispense_money()
            return True
        else:
            print("Ошибка: Недостаточно средств или наличных в банкомате!")
            self.operation_failure()
            return False

    def execute_deposit(self, amount):
        if self.state != 'executing':
            print(f"Ошибка: Банкомат не готов к выполнению операции! Текущее состояние: {self.state}")
            return False

        self.amount = amount
        print(f"Запрошено пополнение на: {amount} руб.")
        print("Внесите купюры в купюроприемник...")
        self.deposit_selected()
        self.receive_money()
        return True

    def dispense_money(self):
        if self.state != 'money_issuing':
            print(f"Ошибка: Банкомат не в состоянии выдачи денег! Текущее состояние: {self.state}")
            return

        print(f"Выдача {self.amount} руб...")
        time.sleep(2)

        if self.simulate_dispense():
            print("Деньги выданы. Заберите их.")
            self.money_dispensed()
            self.return_card()
        else:
            print("Ошибка при выдаче денег!")
            self.dispense_failure()

    def receive_money(self):
        if self.state != 'money_receiving':
            print(f"Ошибка: Банкомат не в состоянии приема денег! Текущее состояние: {self.state}")
            return

        print("Ожидание внесения купюр...")
        time.sleep(3)

        if self.simulate_receive():
            self.balance += self.amount
            print(f"Деньги приняты. Баланс обновлен: {self.balance} руб.")
            self.money_received()
            self.return_card()
        else:
            print("Ошибка при приеме денег!")
            self.receive_failure()

    def simulate_dispense(self):
        return True

    def simulate_receive(self):
        return True

    def return_card(self):
        if self.state != 'card_returning':
            print(f"Ошибка: Банкомат не в состоянии возврата карты! Текущее состояние: {self.state}")
            return

        print("Возврат карты...")
        time.sleep(1)

        if self.simulate_return_card():
            print("Карта возвращена. Заберите её.")
            self.card_returned()
            self.complete_transaction()
        else:
            print("Ошибка при возврате карты!")
            self.return_failure()

    def simulate_return_card(self):
        return True

    def complete_transaction(self):
        if self.state != 'completed':
            return

        print("Транзакция завершена. Банкомат готов к следующему клиенту.")
        time.sleep(1)
        self.reset()
        self.reset_sensors()

    def reset_sensors(self):
        self.card_inserted = False
        self.pin_attempts = 0
        self.card_data = None
        self.operation_type = None
        self.amount = 0

    def get_status(self):
        return {
            'state': self.state,
            'card_inserted': self.card_inserted,
            'balance': self.balance,
            'atm_cash': self.atm_cash
        }

In [ ]:
atm = ATM()

atm.insert_card('123456789012')
time.sleep(1)

atm.enter_pin('1234')

atm.select_operation('withdrawal')
time.sleep(1)

atm.execute_withdrawal(5000)

time.sleep(5)

Карта вставлена. Чтение данных...
Карта успешно считана.
Проверка ПИН-кода...
ПИН-код принят.
Выбрана операция: withdrawal
Запрошено снятие: 5000 руб.
Операция одобрена. Новый баланс: 5000 руб.
Выдача 5000 руб...
Деньги выданы. Заберите их.
Возврат карты...
Карта возвращена. Заберите её.
Транзакция завершена. Банкомат готов к следующему клиенту.


In [ ]:
atm.insert_card('1234567890')

atm.enter_pin('1234')

atm.select_operation('deposit')

atm.execute_deposit(2000)

Карта вставлена. Чтение данных...
Карта успешно считана.
Проверка ПИН-кода...
ПИН-код принят.
Выбрана операция: deposit
Запрошено пополнение на: 2000 руб.
Внесите купюры в купюроприемник...
Ожидание внесения купюр...
Деньги приняты. Баланс обновлен: 7000 руб.
Возврат карты...
Карта возвращена. Заберите её.
Транзакция завершена. Банкомат готов к следующему клиенту.


True